# DEMO 2: Building Visualizations from Different Data Sources

This demo builds on Demo 1. We’ll create **the same set of dashboard visualizations** using each of our four data architecture approaches, so you can see exactly how the SQL differs and what trade-offs emerge in practice.

### What We’ll Build (4 Common Widget Types)
| Widget | Business Question | Chart Type |
| --- | --- | --- |
| KPI Cards | What are our headline numbers? | Counter |
| Revenue by Region | How do regions compare? | Bar chart |
| Monthly Trend | How is revenue trending? | Line chart |
| Top Products | Which products drive the most revenue? | Table |

For **each widget**, we’ll write the dataset query using:
1. Star Schema (JOINs)
2. Gold Table (flat)
3. SQL View (pre-shaped)
4. Metric View (MEASURE syntax)

> **Prerequisite:** Run the Setup notebook and complete Demo 1 first.

---

In [0]:
%run ./Setup

In [0]:
%sql
-- ============================================================
-- WIDGET 1: KPI COUNTERS — "What are our headline numbers?"
-- ============================================================
-- Counter widgets need a SINGLE ROW with one value per metric.
-- Widget type: Counter | Power BI equivalent: Card visual

-- APPROACH A: Star Schema
SELECT
  ROUND(SUM(f.net_revenue), 0)            AS total_revenue,
  COUNT(f.order_id)                       AS total_orders,
  ROUND(AVG(f.net_revenue), 2)            AS avg_order_value,
  ROUND(SUM(f.net_revenue - f.cost), 0)   AS total_profit
FROM fact_sales f;

-- APPROACH B: Gold Table (simplest — no JOINs)
-- SELECT
--   ROUND(SUM(net_revenue), 0)   AS total_revenue,
--   COUNT(order_id)              AS total_orders,
--   ROUND(AVG(net_revenue), 2)   AS avg_order_value,
--   ROUND(SUM(profit), 0)        AS total_profit
-- FROM gold_sales;

-- APPROACH C: SQL View (pre-aggregated, just sum the monthly totals)
-- SELECT
--   ROUND(SUM(total_revenue), 0)   AS total_revenue,
--   SUM(order_count)               AS total_orders,
--   ROUND(SUM(total_profit), 0)    AS total_profit
-- FROM vw_monthly_revenue;

-- APPROACH D: Metric View (centralized definitions)
-- SELECT
--   MEASURE(`Total Revenue`)  AS total_revenue,
--   MEASURE(`Order Count`)    AS total_orders,
--   MEASURE(`Avg Order Value`) AS avg_order_value,
--   MEASURE(`Total Profit`)   AS total_profit
-- FROM mv_sales_metrics
-- GROUP BY ALL;

In [0]:
%sql
-- ============================================================
-- WIDGET 2: REVENUE BY REGION BAR CHART
-- ============================================================
-- Bar charts need: one categorical column (X) + one numeric column (Y).
-- Widget config: Visualization type = Bar, X = region, Y = total_revenue

-- APPROACH A: Star Schema (requires JOIN to dim_region)
SELECT
  r.region_name                    AS region,
  r.territory,
  ROUND(SUM(f.net_revenue), 2)     AS total_revenue,
  COUNT(f.order_id)                AS order_count
FROM fact_sales f
JOIN dim_region r ON f.region_id = r.region_id
GROUP BY r.region_name, r.territory
ORDER BY total_revenue DESC;

-- APPROACH B: Gold Table
-- SELECT
--   region,
--   territory,
--   ROUND(SUM(net_revenue), 2) AS total_revenue,
--   COUNT(order_id)            AS order_count
-- FROM gold_sales
-- GROUP BY region, territory
-- ORDER BY total_revenue DESC;

-- APPROACH C: SQL View (already aggregated!)
-- SELECT region_name, territory, total_revenue, total_orders
-- FROM vw_region_performance
-- ORDER BY total_revenue DESC;

-- APPROACH D: Metric View
-- SELECT `Region`, `Territory`,
--   MEASURE(`Total Revenue`) AS total_revenue,
--   MEASURE(`Order Count`)   AS order_count
-- FROM mv_sales_metrics
-- GROUP BY ALL
-- ORDER BY total_revenue DESC;

In [0]:
%sql
-- ============================================================
-- WIDGET 3: MONTHLY REVENUE TREND (LINE CHART)
-- ============================================================
-- Line charts need: one date/time column (X) + one numeric column (Y).
-- Widget config: Visualization type = Line, X = order_month, Y = total_revenue

-- APPROACH A: Star Schema (JOIN to dim_date for month grouping)
SELECT
  DATE_TRUNC('MONTH', f.order_date) AS order_month,
  ROUND(SUM(f.net_revenue), 2)      AS total_revenue,
  ROUND(SUM(f.net_revenue - f.cost), 2) AS total_profit,
  COUNT(f.order_id)                 AS order_count
FROM fact_sales f
GROUP BY DATE_TRUNC('MONTH', f.order_date)
ORDER BY order_month;

-- APPROACH B: Gold Table
-- SELECT
--   DATE_TRUNC('MONTH', order_date) AS order_month,
--   ROUND(SUM(net_revenue), 2)      AS total_revenue,
--   ROUND(SUM(profit), 2)           AS total_profit,
--   COUNT(order_id)                 AS order_count
-- FROM gold_sales
-- GROUP BY DATE_TRUNC('MONTH', order_date)
-- ORDER BY order_month;

-- APPROACH C: SQL View (already monthly! Just select.)
-- SELECT order_month, total_revenue, total_profit, order_count
-- FROM vw_monthly_revenue
-- WHERE region = 'Northeast'   -- optional filter
-- ORDER BY order_month;

-- APPROACH D: Metric View
-- SELECT
--   `Order Month`,
--   MEASURE(`Total Revenue`)  AS total_revenue,
--   MEASURE(`Total Profit`)   AS total_profit,
--   MEASURE(`Order Count`)    AS order_count
-- FROM mv_sales_metrics
-- GROUP BY ALL
-- ORDER BY `Order Month` ASC;

In [0]:
%sql
-- ============================================================
-- WIDGET 4: TOP PRODUCTS TABLE
-- ============================================================
-- Tables need: multiple columns returned. The widget displays them as rows.
-- Widget config: Visualization type = Table

-- APPROACH A: Star Schema (JOIN to dim_product)
SELECT
  p.product_name,
  p.category,
  p.subcategory,
  ROUND(SUM(f.net_revenue), 2) AS total_revenue,
  SUM(f.quantity)              AS units_sold,
  ROUND((SUM(f.net_revenue) - SUM(f.cost)) / SUM(f.net_revenue) * 100, 1) AS profit_margin_pct
FROM fact_sales f
JOIN dim_product p ON f.product_id = p.product_id
GROUP BY p.product_name, p.category, p.subcategory
ORDER BY total_revenue DESC
LIMIT 10;

-- APPROACH B: Gold Table
-- SELECT
--   product_name,
--   product_category,
--   ROUND(SUM(net_revenue), 2)  AS total_revenue,
--   SUM(quantity)               AS units_sold,
--   ROUND(SUM(profit) / SUM(net_revenue) * 100, 1) AS profit_margin_pct
-- FROM gold_sales
-- GROUP BY product_name, product_category
-- ORDER BY total_revenue DESC
-- LIMIT 10;

-- APPROACH C: SQL View (pre-ranked!)
-- SELECT product_name, category, total_revenue, units_sold, profit_margin_pct
-- FROM vw_top_products
-- WHERE revenue_rank <= 10
-- ORDER BY revenue_rank;

-- APPROACH D: Metric View
-- SELECT
--   `Product Name`,
--   `Product Category`,
--   MEASURE(`Total Revenue`)   AS total_revenue,
--   MEASURE(`Total Quantity`)  AS units_sold,
--   MEASURE(`Profit Margin`)   AS profit_margin_pct
-- FROM mv_sales_metrics
-- GROUP BY ALL
-- ORDER BY total_revenue DESC
-- LIMIT 10;

## Practical Guidance: Which Approach for Which Widget?

Now that you’ve seen all four approaches, here’s practical guidance on choosing:

### Best Fit by Widget Type

| Widget Type | Best Approach | Why |
| --- | --- | --- |
| **KPI Counters** | Metric View | Ensures consistent definitions ("revenue" always means the same thing) |
| **Bar/Pie Charts** | Gold Table or Metric View | Simple grouping, no JOIN complexity |
| **Line Charts (trends)** | SQL View | Pre-aggregated time-series avoids costly GROUP BY on large tables |
| **Detail Tables** | Star Schema or Gold Table | Needs multiple columns, flexibility matters |
| **Drill-through pages** | Star Schema | Maximum ad-hoc flexibility for detail exploration |

### Performance Considerations

| Approach | First Load | Refresh Behavior | Caching |
| --- | --- | --- | --- |
| Star Schema | Moderate (JOINs) | Always fresh | Dashboard result cache |
| Gold Table | Fastest | Stale until ETL reruns | Dashboard result cache |
| SQL View | Varies (complex views slower) | Always fresh | Dashboard result cache |
| Metric View | Good (optimizable) | Always fresh | Can be materialized |

### Mixing Approaches in One Dashboard (Recommended)

A real-world dashboard typically uses **multiple approaches**:
- **Page 1 (Executive Summary):** Metric View for KPIs (consistent definitions)
- **Page 2 (Trends):** SQL View for time-series (pre-shaped)
- **Page 3 (Detail/Drill):** Star Schema for ad-hoc exploration
- **High-traffic widgets:** Gold Table for sub-second response

---

In [0]:
%sql
-- ============================================================
-- BONUS: Multi-Series Visualizations
-- ============================================================
-- Adding a "Color" / "Series" dimension creates grouped or stacked charts.
-- Widget config: Set the Color field to a categorical column.

-- Star Schema: Revenue by Region, split by Channel
SELECT
  r.region_name        AS region,
  f.channel,
  ROUND(SUM(f.net_revenue), 2) AS total_revenue
FROM fact_sales f
JOIN dim_region r ON f.region_id = r.region_id
GROUP BY r.region_name, f.channel
ORDER BY region, total_revenue DESC;

-- Metric View equivalent:
-- SELECT `Region`, `Channel`,
--   MEASURE(`Total Revenue`) AS total_revenue
-- FROM mv_sales_metrics
-- GROUP BY ALL
-- ORDER BY `Region`, total_revenue DESC;

## Steps: Building the Dashboard in the UI

### Step 1: Create a new dashboard
1. Click **+ New** > **Dashboard** in the left sidebar
2. Name it: **Sales Performance Dashboard**

### Step 2: Add datasets (Data tab)
For each widget above, add a dataset:
1. Click **Data** tab > **+ Add data** > **Create from SQL**
2. Paste the SQL from the approach you chose
3. Click **Run** to verify results
4. Name the dataset descriptively (e.g., "KPI Summary", "Revenue by Region")

### Step 3: Build widgets (Canvas tab)
1. Click **Canvas** tab > **Add a widget** > **Visualization**
2. Select the dataset from the dropdown
3. Choose the visualization type (Counter, Bar, Line, Table)
4. Map columns to axes (X, Y, Color)
5. Add a title and adjust formatting

### Step 4: Layout and pages
- **Drag** widgets to position; **resize** by dragging corners
- Widgets **snap to grid** for alignment
- Use **+ Add page** for multi-page dashboards:
  - Page 1: Executive KPIs (Metric View datasets)
  - Page 2: Regional Analysis (Gold Table datasets)
  - Page 3: Product Deep-Dive (Star Schema datasets)

### Step 5: Publish
- Click **Publish** to make the dashboard available to viewers
- Published dashboards auto-refresh on configurable schedules

---

---

## Summary

### What We Built
| Widget | Star Schema SQL | Gold Table SQL | View SQL | Metric View SQL |
| --- | --- | --- | --- | --- |
| KPI Counters | SUM + no JOIN | SUM, simplest | SUM of view | MEASURE() |
| Bar Chart | JOIN + GROUP BY | GROUP BY only | SELECT from view | MEASURE + GROUP BY ALL |
| Line Chart | GROUP BY date | GROUP BY date | SELECT (pre-grouped) | MEASURE + Order Month |
| Top Products | JOIN + LIMIT | GROUP BY + LIMIT | WHERE rank <= N | MEASURE + LIMIT |

### Key Insight
The **same business question** can be answered with any approach — the difference is:
- **Who does the work**: ETL (gold table), view author (views), query author (star schema), or metric owner (metric view)
- **When the work happens**: At ETL time, at view creation, or at query time
- **Who benefits**: Performance-critical dashboards (gold), governed metrics (metric view), flexible exploration (star schema)

**Next:** In Demo 3, we’ll add interactivity — filters, parameters, cross-filtering — and see how each data architecture handles dynamic user input.